# NYC Property Sales: Data Cleaning

This notebook loads the consolidated dataset and applies cleaning fixes identified during the consolidation step.

Issues addressed here:
1. Blank strings used as missing values in older files (pre-2007) - standardized to `NULL`
2. Whitespace in string columns - trimmed
3. `Ease-Ment` column - dropped (empty across all 85 files)
4. Column types - all columns are strings from CSV ingestion; numeric and date columns cast to correct types

 *Note on $0 and null Sale Prices:* These are kept in the cleaned dataset. $0 sales represent non-arm's-length transfers (gifts, internal transfers, deed corrections).

## Imports

In [94]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pathlib import Path
from pyspark.sql.types import StringType


## Load Consolidated Data

In [95]:
spark = SparkSession.builder.appName("nyc-sales-cleaning").getOrCreate()

data_path = Path("data/consolidated_nyc_sales.parquet")

df = spark.read.parquet(str(data_path.resolve()))

df.show(5)

print(f"Rows loaded: {df.count()}")
print(f"Columns ({len(df.columns)}): {df.columns}")

+-------+------------------+-----------------------+--------------------+-----+---+---------+-------------------------+-------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+-------------------+----+------------+
|Borough|      Neighborhood|Building Class Category|Tax Class At Present|Block|Lot|Ease-Ment|Building Class At Present|            Address|Apartment Number|Zip Code|Residential Units|Commercial Units|Total Units|Land Square Feet|Gross Square Feet|Year Built|Tax Class At Time Of Sale|Building Class At Time Of Sale|Sale Price|          Sale Date|Year|Borough_Name|
+-------+------------------+-----------------------+--------------------+-----+---+---------+-------------------------+-------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+------------

## Pre-Cleaning Analysis

Before making any changes, we document the state of the data - null counts, blank string counts, the Ease-Ment column, and the Sale Price distribution.

This gives us a baseline and confirms each issue before fixing it.

### Schema

In [96]:
df.printSchema()

root
 |-- Borough: string (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Building Class Category: string (nullable = true)
 |-- Tax Class At Present: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- Lot: string (nullable = true)
 |-- Ease-Ment: string (nullable = true)
 |-- Building Class At Present: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Apartment Number: string (nullable = true)
 |-- Zip Code: string (nullable = true)
 |-- Residential Units: string (nullable = true)
 |-- Commercial Units: string (nullable = true)
 |-- Total Units: string (nullable = true)
 |-- Land Square Feet: string (nullable = true)
 |-- Gross Square Feet: string (nullable = true)
 |-- Year Built: string (nullable = true)
 |-- Tax Class At Time Of Sale: string (nullable = true)
 |-- Building Class At Time Of Sale: string (nullable = true)
 |-- Sale Price: string (nullable = true)
 |-- Sale Date: string (nullable = true)
 |-- Year: integer (nullabl

### NULL Counts Per Column

In [97]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0--------------------------------
 Borough                        | 26580  
 Neighborhood                   | 26580  
 Building Class Category        | 26580  
 Tax Class At Present           | 26824  
 Block                          | 26580  
 Lot                            | 26580  
 Ease-Ment                      | 194282 
 Building Class At Present      | 26824  
 Address                        | 26580  
 Apartment Number               | 157516 
 Zip Code                       | 26596  
 Residential Units              | 43806  
 Commercial Units               | 43806  
 Total Units                    | 43806  
 Land Square Feet               | 43807  
 Gross Square Feet              | 43806  
 Year Built                     | 32607  
 Tax Class At Time Of Sale      | 26580  
 Building Class At Time Of Sale | 26580  
 Sale Price                     | 26580  
 Sale Date                      | 26580  
 Year                           | 0      
 Borough_Name                   | 

In [98]:
df.filter(F.col("Borough").isNull()).show(5, truncate=False)

+-------+------------+-----------------------+--------------------+-----+----+---------+-------------------------+-------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+---------+----+------------+
|Borough|Neighborhood|Building Class Category|Tax Class At Present|Block|Lot |Ease-Ment|Building Class At Present|Address|Apartment Number|Zip Code|Residential Units|Commercial Units|Total Units|Land Square Feet|Gross Square Feet|Year Built|Tax Class At Time Of Sale|Building Class At Time Of Sale|Sale Price|Sale Date|Year|Borough_Name|
+-------+------------+-----------------------+--------------------+-----+----+---------+-------------------------+-------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+---------+----+

The dataframe read cells which did not contain any data, dropping these rows

In [99]:
df = df.filter(F.col("Borough").isNotNull())

In [100]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show(vertical=True)

-RECORD 0--------------------------------
 Borough                        | 0      
 Neighborhood                   | 0      
 Building Class Category        | 0      
 Tax Class At Present           | 244    
 Block                          | 0      
 Lot                            | 0      
 Ease-Ment                      | 167702 
 Building Class At Present      | 244    
 Address                        | 0      
 Apartment Number               | 130936 
 Zip Code                       | 16     
 Residential Units              | 17226  
 Commercial Units               | 17226  
 Total Units                    | 17226  
 Land Square Feet               | 17227  
 Gross Square Feet              | 17226  
 Year Built                     | 6027   
 Tax Class At Time Of Sale      | 0      
 Building Class At Time Of Sale | 0      
 Sale Price                     | 0      
 Sale Date                      | 0      
 Year                           | 0      
 Borough_Name                   | 

**OBSERVATION:**

- **26,580 fully empty rows:** trailing blank rows from source CSVs. Dropped on Borough.isNotNull().
- **Ease-Ment:** 167,702 nulls. Column exists in all files but has no meaningful data. 
- **Apartment Number:** 130,936 nulls. Expected, most properties are not apartments.
- **Units / Square Feet columns:** ~17,226 nulls each. Likely properties with no size or unit data recorded.
- **Year Built:** 6,027 nulls. Some properties have no recorded build year.
- **Tax Class / Building Class At Present:** 244 nulls. Small data entry gaps in source files.

### Blank String Counts Per Column

Some Pre-2007 files use blank or whitespace strings instead of NULL. Counting blank/whitespace-only strings per string column to see how widespread this is.

In [101]:
#Changing name for convenience and avoiding errors
df = df.withColumnRenamed("Ease-Ment", "EaseMent")

string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]

blank_counts = df.select(
    [F.count(F.when(F.trim(F.col(c)) == "", c)).alias(c) for c in string_cols]
)

blank_counts.show(vertical=True)

-RECORD 0---------------------------------
 Borough                        | 0       
 Neighborhood                   | 0       
 Building Class Category        | 1481    
 Tax Class At Present           | 20640   
 Block                          | 0       
 Lot                            | 0       
 EaseMent                       | 1432447 
 Building Class At Present      | 20640   
 Address                        | 7       
 Apartment Number               | 1119671 
 Zip Code                       | 0       
 Residential Units              | 0       
 Commercial Units               | 0       
 Total Units                    | 0       
 Land Square Feet               | 0       
 Gross Square Feet              | 0       
 Year Built                     | 0       
 Tax Class At Time Of Sale      | 0       
 Building Class At Time Of Sale | 0       
 Sale Price                     | 0       
 Sale Date                      | 0       
 Borough_Name                   | 0       



** OBSERVATIONS:**

- **Ease-Ment:** 1,432,447 blank strings + 167,702 nulls = covers essentially the entire dataset. Can drop the column.
- **Apartment Number:** 1,119,671 blank strings + 130,936 nulls. Most properties have no apartment number, recorded inconsistently across files.
- **Tax Class / Building Class At Present:** 20,640 blanks + 244 nulls. Same columns, missing values split between blank and null depending on source file. Blank to null standardization needed.
- **Building Class Category:** 1,481 blanks. Small but standardized.
- **Address:** 7 blanks. Negligible.

### Ease-Ment Column

Checking whether `Ease-Ment` has any non-null, non-blank values across all rows.

In [102]:
df.filter(F.col("EaseMent").isNotNull() & (F.trim(F.col("EaseMent")) != "")).count()

13

In [103]:
df.filter(F.trim(F.col("EaseMent")) != "").select("EaseMent").show(13,truncate = False)

+--------+
|EaseMent|
+--------+
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
|E       |
+--------+



### Sale Price Distribution

Checking the breakdown of null, $0, and positive sale prices before any filtering.

In [104]:
df.groupBy(
    F.when(F.col("Sale Price").isNull(), "NULL")
     .when(F.col("Sale Price").cast("double") == 0, "$0")
     .otherwise("> $0")
     .alias("Sale Price Category")
).count().orderBy("count", ascending=False).show()

+-------------------+-------+
|Sale Price Category|  count|
+-------------------+-------+
|               > $0|1116476|
|                 $0| 483686|
+-------------------+-------+



In [105]:
df.filter(F.col("Sale Price").cast("double") == 0) \
  .groupBy("Building Class At Time Of Sale") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(20)

+------------------------------+-----+
|Building Class At Time Of Sale|count|
+------------------------------+-----+
|                            R4|46996|
|                            A1|40782|
|                            B1|37543|
|                            C0|36995|
|                            B2|33703|
|                            A5|29970|
|                            B3|24883|
|                            A2|19220|
|                            D4|17613|
|                            R5|16915|
|                            V0|14721|
|                            B9|11162|
|                            C3| 9288|
|                            A9| 8549|
|                            C1| 8115|
|                            C2| 7765|
|                            S2| 7564|
|                            R3| 6303|
|                            V1| 6154|
|                            RG| 5025|
+------------------------------+-----+
only showing top 20 rows


483,686 rows (30% of dataset) have a Sale Price of $0. These are spread across all residential building classes with no concentration in any single type, which suggests these may be gifts, inheritances, or internal company transfers rather than actual market sales.

### Sale Price - Basic Statistics

After casting to double, get a quick summary to check for outliers and confirm the column looks right.

In [106]:
df.select(F.col("Sale Price").cast("double").alias("Sale Price")).describe().show()

+-------+--------------------+
|summary|          Sale Price|
+-------+--------------------+
|  count|             1600162|
|   mean|   854580.7273132346|
| stddev|1.1633562012890184E7|
|    min|                 0.0|
|    max|       4.111111766E9|
+-------+--------------------+



### Sample Rows

Quick look at the raw data before cleaning.

In [107]:
df.show(5, truncate=False)

+-------+------------------+-------------------------------------------+--------------------+-----+---+--------+-------------------------+-------------------+----------------+--------+-----------------+----------------+-----------+----------------+-----------------+----------+-------------------------+------------------------------+----------+-------------------+----+------------+
|Borough|Neighborhood      |Building Class Category                    |Tax Class At Present|Block|Lot|EaseMent|Building Class At Present|Address            |Apartment Number|Zip Code|Residential Units|Commercial Units|Total Units|Land Square Feet|Gross Square Feet|Year Built|Tax Class At Time Of Sale|Building Class At Time Of Sale|Sale Price|Sale Date          |Year|Borough_Name|
+-------+------------------+-------------------------------------------+--------------------+-----+---+--------+-------------------------+-------------------+----------------+--------+-----------------+----------------+-----------+-

## 1. Trim Whitespace

String columns in the older files contain trailing whitespace. Trimming all string columns before replacing blanks with NULL.

In [108]:
for col_name in string_cols:
    df = df.withColumn(col_name, F.trim(F.col(col_name)))

print(f"Trimmed {len(string_cols)} string columns")

Trimmed 22 string columns


## 2. Standardize Blank Strings to NULL

After trimming, blank strings are now empty strings `""`. Replacing with `NULL` so missing values are consistent across all years.

In [ ]:
for col_name in string_cols:
    df = df.withColumn(col_name,F.when(F.col(col_name) == "", None).otherwise(F.col(col_name)))



Blank strings replaced with NULL


In [115]:
blank_counts = df.select(
    [F.count(F.when(F.trim(F.col(c)) == "", c)).alias(c) for c in string_cols]
)

blank_counts.show(vertical=True)

-RECORD 0-----------------------------
 Borough                        | 0   
 Neighborhood                   | 0   
 Building Class Category        | 0   
 Tax Class At Present           | 0   
 Block                          | 0   
 Lot                            | 0   
 EaseMent                       | 0   
 Building Class At Present      | 0   
 Address                        | 0   
 Apartment Number               | 0   
 Zip Code                       | 0   
 Residential Units              | 0   
 Commercial Units               | 0   
 Total Units                    | 0   
 Land Square Feet               | 0   
 Gross Square Feet              | 0   
 Year Built                     | 0   
 Tax Class At Time Of Sale      | 0   
 Building Class At Time Of Sale | 0   
 Sale Price                     | 0   
 Sale Date                      | 0   
 Borough_Name                   | 0   



## 3. Drop Ease-Ment Column

Confirmed above: `Ease-Ment` has zero non-null, non-blank values across all rows. Dropping it.

In [117]:
df = df.drop("EaseMent")

print(f"Columns after drop: {len(df.columns)}")

Columns after drop: 22


## 4. Cast Column Types

All columns arrive as strings from CSV ingestion. Casting numeric and date columns to correct types.

In [118]:
df = (df
    .withColumn("Sale Price", F.col("Sale Price").cast("double"))
    .withColumn("Residential Units", F.col("Residential Units").cast("integer"))
    .withColumn("Commercial Units", F.col("Commercial Units").cast("integer"))
    .withColumn("Total Units", F.col("Total Units").cast("integer"))
    .withColumn("Land Square Feet", F.col("Land Square Feet").cast("double"))
    .withColumn("Gross Square Feet", F.col("Gross Square Feet").cast("double"))
    .withColumn("Year Built", F.col("Year Built").cast("integer"))
    .withColumn("Zip Code", F.col("Zip Code").cast("integer"))
    .withColumn("Borough", F.col("Borough").cast("integer"))
    .withColumn("Sale Date", F.to_date(F.col("Sale Date")))
)

df.printSchema()

root
 |-- Borough: integer (nullable = true)
 |-- Neighborhood: string (nullable = true)
 |-- Building Class Category: string (nullable = true)
 |-- Tax Class At Present: string (nullable = true)
 |-- Block: string (nullable = true)
 |-- Lot: string (nullable = true)
 |-- Building Class At Present: string (nullable = true)
 |-- Address: string (nullable = true)
 |-- Apartment Number: string (nullable = true)
 |-- Zip Code: integer (nullable = true)
 |-- Residential Units: integer (nullable = true)
 |-- Commercial Units: integer (nullable = true)
 |-- Total Units: integer (nullable = true)
 |-- Land Square Feet: double (nullable = true)
 |-- Gross Square Feet: double (nullable = true)
 |-- Year Built: integer (nullable = true)
 |-- Tax Class At Time Of Sale: string (nullable = true)
 |-- Building Class At Time Of Sale: string (nullable = true)
 |-- Sale Price: double (nullable = true)
 |-- Sale Date: date (nullable = true)
 |-- Year: integer (nullable = true)
 |-- Borough_Name: string (

## Post-Cleaning Validation

Confirming the cleaning steps applied correctly - null counts after standardization, row count unchanged, and schema looks right.

In [ ]:
print(f"Total rows: {df.count()}")
print(f"Total columns: {len(df.columns)}")

# NULL counts per column after cleaning
df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns]).show(vertical=True)

# Sale Price breakdown 
df.groupBy(
    F.when(F.col("Sale Price").isNull(), "NULL")
     .when(F.col("Sale Price") == 0, "$0")
     .otherwise("> $0")
     .alias("Sale Price Category")
).count().orderBy("count", ascending=False).show()

df.show(5)

Total rows: 1600162
Total columns: 22


-RECORD 0---------------------------------
 Borough                        | 0       
 Neighborhood                   | 0       
 Building Class Category        | 1481    
 Tax Class At Present           | 20884   
 Block                          | 0       
 Lot                            | 0       
 Building Class At Present      | 20884   
 Address                        | 7       
 Apartment Number               | 1250607 
 Zip Code                       | 16      
 Residential Units              | 17226   
 Commercial Units               | 17226   
 Total Units                    | 17226   
 Land Square Feet               | 17227   
 Gross Square Feet              | 17226   
 Year Built                     | 6027    
 Tax Class At Time Of Sale      | 0       
 Building Class At Time Of Sale | 0       
 Sale Price                     | 0       
 Sale Date                      | 0       
 Year                           | 0       
 Borough_Name                   | 0       

+---------

## Save Cleaned Data

In [120]:
output_path = "data/nyc_property_sales_cleaned"

df.write.mode("overwrite").parquet(output_path)

print(f"Saved to: {output_path}")

26/10/22 17:05:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/22 17:05:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/22 17:05:49 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 76.00% for 10 writers
26/10/22 17:05:50 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/22 17:05:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


Saved to: data/nyc_property_sales_cleaned
